In [0]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from pyspark.sql.window import Window # 如果需要复杂的去重逻辑备用

class NYC_Taxi_Silver_Loader:
    def __init__(self, spark, run_id):
        self.spark = spark
        self.run_id = run_id
        
        # 优化：将业务规则封装，便于后续作为配置文件(如 JSON/YAML)动态加载
        self.BASE_RULES = {
            "missing_pickup": F.col("pickup_datetime").isNull(),
            "missing_dropoff": F.col("dropoff_datetime").isNull(),
            "dropoff_before_pickup": (F.col("pickup_datetime").isNotNull()) & 
                                     (F.col("dropoff_datetime").isNotNull()) & 
                                     (F.col("dropoff_datetime") < F.col("pickup_datetime")),
            "passenger_count_invalid": (F.col("passenger_count") < 0) | (F.col("passenger_count") > 9),
            "total_amount_negative": F.col("total_amount") < 0,
            "invalid_YYYYMM": (F.col("YYYYMM") < 190001) | (F.col("YYYYMM") > 300012),
            "duration_out_of_range": (F.col("duration_min") < 2.0) | (F.col("duration_min") > 180),
            "distance_too_small": F.col("trip_distance") <= 0.1,
            "efficiency_too_high": F.col("temp_eff") >= 15.0,
            "fare_too_low": F.col("fare_amount") <= 2.5
        }

    def process(self, bronze_df: DataFrame) -> None:
        """
        主处理流程：特征提取 -> 规则校验 -> 持久化缓存 -> 数据分流写入
        """
        
        # --- 1. 特征提取与 Audit (审计) 元数据注入 ---
        # 最佳实践：使用 select 替代多个连续的 withColumn 以优化 Catalyst 执行计划
        enriched_df = bronze_df.select(
            "*",
            ((F.col("dropoff_datetime").cast("long") - F.col("pickup_datetime").cast("long")) / 60.0).alias("duration_min"),
            # 注入数据血缘和审计字段
            F.lit(self.run_id).alias("_run_id"),
            F.current_timestamp().alias("_processed_at")
        ).withColumn(
            "temp_eff", 
            F.when(F.col("duration_min") > 0, F.col("fare_amount") / F.col("duration_min")).otherwise(F.lit(0))
        )

        # --- 2. 内存级规则校验 (避免 Schema 污染) ---
        # 最佳实践：直接在数组中进行条件判断，不需要生成中间列，避免后续 drop 操作
        rule_evaluations = [
            F.when(condition, F.lit(rule_name)) 
            for rule_name, condition in self.BASE_RULES.items()
        ]

        dq_df = enriched_df.withColumn(
            "violated_rules", 
            F.array_remove(F.array(*rule_evaluations), None)
        ).withColumn(
            "is_valid", 
            F.size(F.col("violated_rules")) == 0
        )

        # --- 3. 缓存 DataFrame (极度重要) ---
        # 因为后续我们需要将数据分流（filter 两次），且在写入前需要执行 collect() 获取分区
        # 如果不 persist，上游的读取、特征计算和 DQ 校验会被触发重复计算 (DAG 重算)
        dq_df.persist()

        try:
            # --- 4. 数据分流 (Data Splitting) ---
            valid_df = dq_df.filter("is_valid").drop("violated_rules", "is_valid")
            rejected_df = dq_df.filter("~is_valid")

            # --- 5. 幂等覆盖写入 ---
            self._write_to_delta(valid_df, "process_silver.silver_nyc_taxi")
            self._write_to_delta(rejected_df, "process_silver.silver_nyc_taxi_quarantine")
            
        finally:
            # 确保释放内存资源
            dq_df.unpersist()

    def _write_to_delta(self, df: DataFrame, table_name: str):
        # 此时 df 已经被 persist，这里的 collect() 是极速的，只扫描内存中的数据
        partitions_rows = df.select("YYYYMM").distinct().collect()
        
        if not partitions_rows:
            print(f"No data to write for {table_name}. Skipping.")
            return

        partitions = [str(r["YYYYMM"]) for r in partitions_rows]
        replace_cond = f"YYYYMM IN ({','.join(partitions)})"
        
        # 使用 Spark 3.x / Delta Lake 的原生动态分区覆盖也是一种选择
        # 但在生产环境中，显式指定 replaceWhere 更加安全，能防止意料之外的全局覆盖
        df.write \
          .format("delta") \
          .mode("overwrite") \
          .option("replaceWhere", replace_cond) \
          .option("mergeSchema", "true") \
          .saveAsTable(table_name)
        
        # 优化：去掉了原版的 df.count()，因为它会再次触发 Action
        print(f"Successfully loaded data to {table_name} for partitions {partitions}")

In [0]:
from pyspark.sql import SparkSession

# 假设这是你的 Databricks Notebook 或 PySpark 提交脚本的入口
spark = SparkSession.builder.appName("NYCTaxi_Silver_Processing").getOrCreate()

# 1. 定义你的环境参数
BRONZE_TABLE = "nyc.process_bronze.brz_yellow_nyc_taxi"
RUN_ID = "airflow_dag_run_id_12345" # 实际生产中通常由调度工具(如 Airflow/Databricks Workflow)传入
TARGET_YYYYMM = "202310" # 目标处理月份

# 2. 实例化我们刚才优化的 Loader 类
silver_loader = NYC_Taxi_Silver_Loader(spark, run_id=RUN_ID)

# 3. 读取 Bronze 层数据 (按需过滤)
# 最佳实践：不要全表读取（spark.read.table），一定要带上下推过滤(Predicate Pushdown)
bronze_df = spark.table(BRONZE_TABLE).filter(f"YYYYMM = {TARGET_YYYYMM}")

# 4. 执行处理流程
silver_loader.process(bronze_df)

print(f"Pipeline finished successfully for run_id: {RUN_ID}")